# DriveGuard - Milestone 4: Survival / RUL

Builds the survival tables (rolling features + event/duration targets) and benchmarks
classical survival models: Cox PH, Weibull AFT, Random Survival Forest.
Metric: concordance index (primary) + RUL MAE.

**Settings:** Add data `driveguard-backblaze-interim`; Internet On; Secret `GITHUB_TOKEN`.
GPU optional (these models are CPU). ~30-60 min.

In [ ]:
# 1. Clone repo
import os, subprocess
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
URL = f'https://{token}@github.com/keerthirevanth/driveguard-predictive-maintenance.git'
REPO = '/kaggle/working/driveguard-predictive-maintenance'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', URL, REPO], check=True)
import sys; sys.path.insert(0, f'{REPO}/src')
print('cloned:', os.path.exists(REPO))

In [ ]:
# 2. Install deps
!pip install -q polars pyarrow mlflow lifelines scikit-survival 2>/dev/null
print('deps installed')

In [ ]:
# 3. Link dataset
import os, glob
os.makedirs(f'{REPO}/data/interim', exist_ok=True)
os.makedirs(f'{REPO}/data/processed', exist_ok=True)
qf = glob.glob('/kaggle/input/**/data_*.parquet', recursive=True)
sf = glob.glob('/kaggle/input/**/drive_summary.parquet', recursive=True)
assert qf and sf, 'dataset not attached?'
def link(src, dst):
    if os.path.lexists(dst): os.remove(dst)
    os.symlink(src, dst)
for f in qf: link(f, f"{REPO}/data/interim/{os.path.basename(f)}")
link(sf[0], f'{REPO}/data/processed/drive_summary.parquet')
print('interim:', sorted(os.listdir(f'{REPO}/data/interim')))

In [ ]:
# 4. Build survival tables (rolling features + event/duration)
from pathlib import Path
from driveguard.config import load_config
from driveguard.features.survival_data import make_survival_dataset
ROOT = Path(REPO); cfg = load_config(f'{REPO}/config/config.yaml')
info = make_survival_dataset(cfg, ROOT)
print('built:', info['out_dir'])
for k in ['train', 'val', 'test']:
    print(k, info[k]['orig_events'], 'events /', info[k]['orig_censored'], 'censored')

In [ ]:
# 5. Survival bake-off
import json
from driveguard.models.survival import run_survival
FDIR = f'{REPO}/data/processed/survival_rolling'
board = run_survival(FDIR, ['cox_ph', 'weibull_aft', 'random_survival_forest'],
                     train_cap=120000, eval_cap=80000,
                     mlflow_uri='/kaggle/working/mlruns')
json.dump(board, open('/kaggle/working/survival_leaderboard.json', 'w'), indent=2)
print('survival bake-off complete')

In [ ]:
# 6. Leaderboard
import pandas as pd
rows = []
for r in board:
    if r.get('status') == 'ok':
        t = r['test']
        rows.append({'model': r['model'], 'c_index': round(t['c_index'], 4),
                     'rul_mae_days': round(t.get('rul_mae_days') or 0, 1),
                     'events': t['events'], 'fit_sec': r.get('fit_sec')})
    else:
        rows.append({'model': r['model'], 'c_index': 'ERROR: ' + r.get('error', '')[:60]})
pd.DataFrame(rows).sort_values('c_index', ascending=False)

In [ ]:
# 7. Package artifacts
import shutil, os
if os.path.isdir('/kaggle/working/mlruns'):
    shutil.make_archive('/kaggle/working/mlruns_export', 'zip',
                        root_dir='/kaggle/working', base_dir='mlruns')
!ls -lh /kaggle/working/*.json